# 03 — Baseline Ladder (M3)

Builds on the findings from `01_eda.ipynb` and the exploratory model in `02_baseline_model.ipynb`,
without modifying either notebook.

Fills the M3 requirement from the project brief: **one shared `evaluate()` helper (5-fold KFold,
fixed seed) and a 3-step baseline ladder, all scored identically** so the numbers are directly
comparable to each other and to whatever final model comes later.

Also applies the two M2 cleaning decisions that notebook 01 identified but didn't yet apply:
- Drop the 57 constant/duplicate binary columns
- Drop the single outlier row (`ID 1770`, `y=265.32` — confirmed against the real dat; treated here as **drop**, the simplest and most standard choice for one extreme
  point in an otherwise tight distribution — flagged clearly below so it's a one-line change if the
  team decides differently).

In [ ]:
import pandas as pd
from sklearn.dummy import DummyRegressor

from src.cleaning import (
    get_categorical_columns, get_binary_columns, get_drop_list,
    drop_unwanted_columns, drop_outlier_row, OUTLIER_ID
)
from src.evaluate import evaluate
from src.baselines import X0GroupMeanRegressor, build_ridge_baseline

## Load and clean

Cleaning decisions are computed from `train.csv` only (never from test), matching the no-leakage
discipline notebook 02 already established for encoding.

In [ ]:
train = pd.read_csv("../data/raw/train.csv")
print(f"Raw train shape: {train.shape}")

drop_list = get_drop_list(train)
print(f"Columns to drop (constant + duplicate): {len(drop_list)}")

train_clean = drop_unwanted_columns(train, drop_list)

# --- Outlier decision: DROP. Change to skip this line to keep the row instead. ---
train_clean = drop_outlier_row(train_clean, OUTLIER_ID)

print(f"Clean train shape (after dropping columns + outlier row): {train_clean.shape}")

In [ ]:
y = train_clean["y"]
X = train_clean.drop(columns=["ID", "y"])

cat_cols = get_categorical_columns(X)
num_cols = [c for c in X.columns if c not in cat_cols]
print(f"cat_cols={len(cat_cols)}, num_cols={len(num_cols)}")

## The baseline ladder

Every model below is scored through the exact same `evaluate()` function — 5-fold cross-validation,
`random_state=42` — so the R2 values are directly comparable to each other.

In [ ]:
results = []

r = evaluate(DummyRegressor(strategy="mean"), X, y)
results.append({"model": "1. Mean baseline", "mean_r2": r["mean_r2"], "std_r2": r["std_r2"]})
print("1. Mean baseline:", r["mean_r2"], "+/-", r["std_r2"])

In [ ]:
r = evaluate(X0GroupMeanRegressor(), X, y)
results.append({"model": "2. X0 group-mean baseline", "mean_r2": r["mean_r2"], "std_r2": r["std_r2"]})
print("2. X0 group-mean baseline:", r["mean_r2"], "+/-", r["std_r2"])

In [ ]:
ridge = build_ridge_baseline(cat_cols, num_cols)
r = evaluate(ridge, X, y)
results.append({"model": "3. Ridge (one-hot categoricals + cleaned binaries)", "mean_r2": r["mean_r2"], "std_r2": r["std_r2"]})
print("3. Ridge baseline:", r["mean_r2"], "+/-", r["std_r2"])

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("../reports/baseline_ladder_results.csv", index=False)
results_df

## Results (5-fold CV, `random_state=42`)

| Model | Mean R2 | Std R2 |
|---|---|---|
| 1. Mean baseline | -0.0025 | 0.0027 |
| 2. X0 group-mean baseline | 0.5870 | 0.0321 |
| 3. Ridge (one-hot + binaries) | 0.5621 | 0.0367 |

## Findings / what this means for the product story

- **The mean baseline correctly scores ~0** — confirms the M1 promise and the `evaluate()` helper
  is wired correctly.
- **`X0` alone explains ~59% of bench-time variation.** A single categorical column, with no model
  at all beyond "look up the average for this group," gets most of the way to what the best public
  leaderboard solutions ever achieved (0.55-0.58). This is the headline finding the EDA predicted.
- **Ridge (using all 311 cleaned binary columns + the categoricals) does not beat the simple X0
  group-mean** (0.562 vs 0.587) on this cleaned, cross-validated setup. The extra 311 columns add
  complexity without adding signal beyond what `X0` alone already captures.
- **This also outperforms the untuned XGBoost model in notebook 02 (R2=0.4493)** — but that number
  isn't directly comparable: notebook 02 used a single 80/20 split (not 5-fold CV), and didn't drop
  the 57 useless columns or the outlier row first. **Next step: rerun XGBoost through this same
  `evaluate()` function, on this same cleaned data, before concluding anything about which model is
  actually best** — right now we're not comparing like for like.
- Product framing: sequencing production by `X0` configuration group is already, on its own, close
  to as informative as anything in this dataset gets. Going meaningfully further likely needs data
  this dataset doesn't have (line ID, shift, operator, date) — this is the error-analysis story for
  M5.

## Fair comparison: Hashim's XGBoost, re-run on the cleaned data

Notebook 02's XGBoost used a single 80/20 split on the *uncleaned* data (57 useless columns still
in, outlier row still in). Re-running the identical model (default `XGBRegressor`, same
`OrdinalEncoder` approach) through this notebook's cleaned data and shared `evaluate()` function,
for a fair, like-for-like comparison.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor

xgb_preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ("num", "passthrough", num_cols),
])
xgb_pipeline = Pipeline([
    ("preprocess", xgb_preprocessor),
    ("xgb", XGBRegressor(random_state=42)),
])

r = evaluate(xgb_pipeline, X, y)
results.append({"model": "4. XGBoost (default params, cleaned data)", "mean_r2": r["mean_r2"], "std_r2": r["std_r2"]})
print("4. XGBoost on cleaned data:", r["mean_r2"], "+/-", r["std_r2"])

results_df = pd.DataFrame(results)
results_df.to_csv("../reports/baseline_ladder_results.csv", index=False)
results_df

## Updated results (5-fold CV, `random_state=42`, cleaned data)

| Model | Mean R2 | Std R2 |
|---|---|---|
| 1. Mean baseline | -0.0025 | 0.0027 |
| 2. `X0` group-mean baseline | **0.5870** | 0.0321 |
| 3. Ridge (one-hot + binaries) | 0.5621 | 0.0367 |
| 4. XGBoost, default params, cleaned data | 0.5031 | 0.0249 |

**Cleaning + proper cross-validation moved XGBoost from 0.4493 (notebook 02's single split, dirty
data) to 0.5031** — a real improvement, confirming the cleaning work was worth doing. But it is
still the *weakest* of the four once measured fairly: both the trivial `X0` group-mean baseline
and plain Ridge beat default XGBoost on this dataset. This is not a sign anything is broken — it's
consistent with the M5 expectation in the project brief that a sophisticated model may not clear a
simple baseline by much, because a large share of the variance here is genuinely irreducible from
these features. Untuned XGBoost, with no work done to prevent it overfitting on 311 sparse binary
columns, appears to be doing exactly that. **Next real step for M5 belongs to whoever owns
modelling: light, careful tuning (not a tuning spiral) and see whether XGBoost can close this gap —
if it can't, that itself is a legitimate, presentable finding.**